In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import sqlalchemy
import configparser
# Импортируем класс из файла db_connection.py
from db_connection import PostgresConnection
# Импортируем класс из файла data_cleaner.py
from data_cleaner import DataCleaner
import logging

# настраивает базовые параметры для главного журнала
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s"
)

In [2]:
cache_path = 'cache/rfm.parquet'

# Создаем объект для работы с базой данных
db_connection = PostgresConnection()

query = 'SELECT * FROM sales'

data = db_connection.execute_query(query, cache_path, force_refresh=True)

INFO | db_connection | Соединение с PostgreSQL успешно установлено.
INFO | db_connection | Выполнение SQL-запроса...
INFO | db_connection | Запрос выполнен за 11.09 сек. Получено 45128 строк.
INFO | db_connection | Результат сохранён в cache/rfm.parquet


In [3]:
cache_path = 'cache/rfm.parquet'

# Создаем объект для работы с базой данных
db_connection = PostgresConnection()

query = 'SELECT * FROM sales'

data = db_connection.execute_query(query, cache_path, force_refresh=False)

INFO | db_connection | Соединение с PostgreSQL успешно установлено.
INFO | db_connection | Используется кэш: cache/rfm.parquet


In [3]:
data.head(3)

,dr_dat,dr_tim,dr_nchk,dr_ndoc,dr_apt,dr_kkm,dr_tdoc,dr_tpay,dr_cdrugs,dr_ndrugs,...,dr_prod,dr_kol,dr_czak,dr_croz,dr_sdisc,dr_cdisc,dr_bcdisc,dr_tabempl,dr_pos,dr_vzak
0,2022-05-01,08:33:09,6251,7001554,7,22553,Розничная реализация,18,20398,"ОТРИВИН 0,1% 10МЛ. №1 НАЗАЛ.СПРЕЙ ФЛ. /ГЛАКСО/...",...,Новартис Консьюмер Хелс С.А.,1.0,146.00,175.0,8.0,9,200010024843,37,1,1
1,2022-05-01,09:03:32,4659,6002284,6,22568,Розничная реализация,18,541023,РАСТОРОПШИ ШРОТ 100Г. ПОР. /ИВАН ДА МАРЬЯ/,...,ИВАН ДА МАРЬЯ ООО,2.0,38.62,67.0,16.0,35,200000000022,27,1,1
2,2022-05-01,09:07:49,6077,2002811,2,22569,Розничная реализация,18,25634,АККУ-ЧЕК ТЕСТ-ПОЛОСКИ АКТИВ №50 [ACCU-CHEK],...,РОШ ДИАБЕТС КЕА ГМБХ,1.0,1107.57,1363.0,0.0,NULL,NULL,25,1,1


In [ ]:
# Отдельный файл db_connection.py

import configparser
import psycopg
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy.engine import URL
from config_loader import load_config
from pathlib import Path
import logging
import time

logger = logging.getLogger(__name__)

class PostgresConnection:

    def __init__(self, config_file="config_rfm.ini"):
        params = load_config(config_file)

        self.url = URL.create(
            drivername="postgresql+psycopg",
            username=params["user"],
            password=params["password"],
            host=params["host"],
            port=params["port"],
            database=params["database"],
        )

        self.engine = create_engine(
            self.url,
            pool_pre_ping=True,
            future=True,
            pool_recycle=3600
        )

        self._check_connection()
        logger.info("Соединение с PostgreSQL успешно установлено.")

    def _check_connection(self):
        try:
            with self.engine.connect() as conn:
                conn.execute(text("SELECT 1"))
        except SQLAlchemyError as e:
            logger.error("Ошибка подключения к PostgreSQL: %s", e)
            raise ConnectionError(
                "Не удалось установить соединение с PostgreSQL."
            ) from e

    def execute_query(self, query, cache_path=None):
        """
        Выполняет SQL-запрос с опциональным кэшированием.
        Если cache_path указан и директория существует — загружает результат из него.
        """

        cache_file = Path(cache_path)
        cache_file.parent.mkdir(parents=True, exist_ok=True)

        if cache_path and Path(cache_path).exists():
            logger.info("Используется кэш: %s", cache_path)
            return pd.read_parquet(cache_file)


        logger.info("Выполнение SQL-запроса...")
        start = time.perf_counter()
        with self.engine.connect() as connection:
            result = pd.read_sql(text(query), connection)
        elapsed = time.perf_counter() - start
        logger.info(
            "Запрос выполнен за %.2f сек. Получено %d строк.",
            elapsed,
            len(result),
        )

        if cache_path:
            result.to_parquet(cache_file)
            logger.info("Результат сохранён в %s", cache_path)

        return result

In [ ]:
# Отдельный файл config_loader.py

import configparser
from pathlib import Path

def load_config(config_file='config_rfm.ini', section='postgresql'):
    """
    Загружает настройки из .ini файла.
    :param config_file: путь к конфигурационному файлу
    :param section: секция (по умолчанию 'postgresql')
    :return: dict с параметрами подключения
    """
    config_path = Path(config_file)
    if not config_path.exists():
        raise FileNotFoundError(f"Конфигурационный файл не найден: {config_path}")

    parser = configparser.ConfigParser()
    parser.read(config_path)

    if section not in parser:
        raise ValueError(f"Секция '{section}' не найдена в {config_file}")

    return dict(parser[section])

In [ ]:
# Отдельный файл config_rfm

[postgresql]
host = 95.163.241.236
database = apteka
user = student
password = ***
port = 5432